In [17]:
# Parámetros: el pipeline los sustituye al llamar al notebook
CIUDAD = "euskadi"
SNAPSHOT = "2026-06-30"

StatementMeta(, 6e5bb2ae-5355-48cc-8022-3c7c72aabf33, 19, Finished, Available, Finished, False)

# nb_01_silver_listings
**Objetivo:** leer `listings` crudos de bronze (Euskadi 2026-06-30), aplicar las reglas de silver con PySpark y validarlas; después se escribirá la tabla Delta en `lh_silver`.
**Entrada:** `lh_bronze/Files/<ciudad>/<fecha>/listings.csv.gz`.
**Reglas:** las acordadas en `docs/manual.md` (Paso 3); aquí se reproducen en PySpark.

## 4a Leer `listings` con Spark
**Qué:** leo el CSV con y sin `multiLine`.
**Por qué:** los textos largos traen saltos de línea dentro de comillas; sin `multiLine`, Spark cuenta cada línea como una fila nueva.
**Resultado:** sin `multiLine` 10.128 filas; con `multiLine` 6.248 filas y 90 columnas, igual que pandas.

In [18]:
ruta = f"Files/{CIUDAD}/{SNAPSHOT}/listings.csv.gz"

df_mal = spark.read.option("header", True).csv(ruta)
df_bien = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(ruta))

print("sin multiLine:", df_mal.count())
print("con multiLine:", df_bien.count(), "| columnas:", len(df_bien.columns))


StatementMeta(, 6e5bb2ae-5355-48cc-8022-3c7c72aabf33, 20, Finished, Available, Finished, False)

sin multiLine: 10128
con multiLine: 6248 | columnas: 90


## 4b Elegir 39 columnas y declarar los tipos
**Qué:** me quedo con 39 columnas y las convierto a entero largo, decimal, fecha o texto.
**Por qué:** tipos fijos entre snapshots y sin datos personales (`host_name`, descripciones).
**Resultado:** 6.248 filas x 39 columnas. Nulos iguales a pandas: `bathrooms` 0,155, `bedrooms` 0,149, `license` 0,118, `beds` 0,115, `first_review` y `last_review` 0,101.

In [19]:
from pyspark.sql import functions as F

ENTERAS = ["id", "host_id", "accommodates", "availability_30", "availability_60",
    "availability_90", "availability_365", "number_of_reviews", "number_of_reviews_ltm",
    "number_of_reviews_l30d", "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes", "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms", "estimated_occupancy_l365d"]
DECIMALES = ["latitude", "longitude", "bathrooms", "bedrooms", "beds", "minimum_nights",
    "maximum_nights", "review_scores_rating", "review_scores_location", "review_scores_value",
    "reviews_per_month", "estimated_revenue_l365d"]
FECHAS = ["last_scraped", "calendar_last_scraped", "first_review", "last_review"]
TEXTOS = ["neighbourhood_cleansed", "neighbourhood_group_cleansed", "property_type",
    "room_type", "price", "license", "has_availability", "host_is_superhost"]

df = df_bien.select(*(ENTERAS + DECIMALES + FECHAS + TEXTOS))
for c in ENTERAS:   df = df.withColumn(c, F.col(c).cast("long"))
for c in DECIMALES: df = df.withColumn(c, F.col(c).cast("double"))
for c in FECHAS:    df = df.withColumn(c, F.to_date(F.col(c)))

print(df.count(), len(df.columns))
nulos = df.select([F.round(F.mean(F.col(c).isNull().cast("int")), 3).alias(c) for c in df.columns])
print(nulos.toPandas().T.sort_values(0, ascending=False).head(6))

StatementMeta(, 6e5bb2ae-5355-48cc-8022-3c7c72aabf33, 21, Finished, Available, Finished, False)

6248 39
                  0
bathrooms     0.155
bedrooms      0.149
license       0.118
beds          0.115
last_review   0.101
first_review  0.101


## 4c-1 Precio, flags y booleanos
**Qué:** creo `price_num`, los flags `price_missing`, `price_outlier` e `is_long_stay`, y paso `t`/`f` a booleano. Si aparece otro valor, falla.
**Por qué:** un flag debe ser `True` o `False`, nunca `NULL` (por eso el `coalesce`); un nulo no es un falso.
**Resultado:** 572 / 1 / 264. `has_availability`: true 6.216, false 6, nulo 26. `host_is_superhost`: true 2.458, false 3.784, nulo 6.

In [20]:
# 1) Fallar rápido si aparece un valor que no sea t/f
for c in ["has_availability", "host_is_superhost"]:
    malos = df.filter(F.col(c).isNotNull() & ~F.col(c).isin("t", "f")).count()
    assert malos == 0, f"Valores inesperados en {c}"

# 2) Transformaciones
df = (df
    .withColumn("price_num", F.regexp_replace("price", "[$,]", "").cast("double"))
    .withColumn("price_missing", F.col("price_num").isNull())
    .withColumn("price_outlier", F.coalesce(F.col("price_num") == 9999, F.lit(False)))
    .withColumn("is_long_stay", F.coalesce(F.col("minimum_nights") >= 28, F.lit(False)))
    .withColumn("has_availability", F.when(F.col("has_availability") == "t", True).when(F.col("has_availability") == "f", False))
    .withColumn("host_is_superhost", F.when(F.col("host_is_superhost") == "t", True).when(F.col("host_is_superhost") == "f", False)))

# 3) Test de regresión contra pandas
df.agg(F.sum(F.col("price_missing").cast("int")).alias("price_missing"),
       F.sum(F.col("price_outlier").cast("int")).alias("price_outlier"),
       F.sum(F.col("is_long_stay").cast("int")).alias("is_long_stay")).show()
df.groupBy("has_availability").count().show()
df.groupBy("host_is_superhost").count().show()

StatementMeta(, 6e5bb2ae-5355-48cc-8022-3c7c72aabf33, 22, Finished, Available, Finished, False)

+-------------+-------------+------------+
|price_missing|price_outlier|is_long_stay|
+-------------+-------------+------------+
|          572|            1|         264|
+-------------+-------------+------------+

+----------------+-----+
|has_availability|count|
+----------------+-----+
|            NULL|   26|
|            true| 6216|
|           false|    6|
+----------------+-----+

+-----------------+-----+
|host_is_superhost|count|
+-----------------+-----+
|             NULL|    6|
|             true| 2458|
|            false| 3784|
+-----------------+-----+



## 4c-2 Reparar acentos rotos
**Qué:** re-decodifico con ISO-8859-1 los nombres de zona que contienen `Ã`.
**Por qué:** Euskadi trae texto UTF-8 mal codificado; solo se toca lo roto, no lo correcto.
**Resultado:** 0 nombres con `Ã`; provincias Guipúzcoa, Vizcaya y Álava con sus acentos.

In [21]:
for c in ["neighbourhood_cleansed", "neighbourhood_group_cleansed"]:
    reparado = F.decode(F.encode(F.col(c), "ISO-8859-1"), "UTF-8")
    df = df.withColumn(c, F.when(F.col(c).contains("Ã"), reparado).otherwise(F.col(c)))

print("con Ã restantes:", df.filter(F.col("neighbourhood_cleansed").contains("Ã")).count())
df.select("neighbourhood_group_cleansed").distinct().show()

StatementMeta(, 6e5bb2ae-5355-48cc-8022-3c7c72aabf33, 23, Finished, Available, Finished, False)

con Ã restantes: 0
+----------------------------+
|neighbourhood_group_cleansed|
+----------------------------+
|                     Vizcaya|
|                       Álava|
|                   Guipúzcoa|
+----------------------------+



## 4c-3 Licencias a categorías
**Qué:** creo `license_status` y `exempt_type`, y elimino `license`.
**Por qué:** segmentar el comp set y no arrastrar texto libre con posibles datos personales.
**Resultado:** `con_registro` 4.179, `exento` 1.232, `sin_dato` 738, `texto_libre` 99.

In [22]:
lic = F.col("license")
exento = F.trim(F.regexp_extract(lic, "Exempt - ([^<]+)", 1))

df = (df
    .withColumn("license_status",
        F.when(lic.isNull(), "sin_dato")
         .when(lic.contains("Exempt"), "exento")
         .when(lic.contains("Basque Country - Regional") | lic.contains("Spain - National"), "con_registro")
         .otherwise("texto_libre"))
    .withColumn("exempt_type", F.when(exento != "", exento))
    .drop("license"))

df.groupBy("license_status").count().show()
df.groupBy("exempt_type").count().orderBy(F.desc("count")).show()
print("license sigue en el DataFrame:", "license" in df.columns)

StatementMeta(, 6e5bb2ae-5355-48cc-8022-3c7c72aabf33, 24, Finished, Available, Finished, False)

+--------------+-----+
|license_status|count|
+--------------+-----+
|   texto_libre|   99|
|  con_registro| 4179|
|        exento| 1232|
|      sin_dato|  738|
+--------------+-----+

+--------------------+-----+
|         exempt_type|count|
+--------------------+-----+
|                NULL| 5036|
|   tourist apartment|  282|
|rural tourism acc...|  242|
|               hotel|  240|
|              hostel|  186|
|     seasonal rental|  156|
|agrotourism accom...|   95|
|       hotel complex|   10|
|  hotel-type listing|    1|
+--------------------+-----+

license sigue en el DataFrame: False


## 4c-4 Comprobaciones de calidad con gravedad
**Qué:** cuento las filas que incumplen 9 reglas de sentido común. Las bloqueantes (ids únicos y flags sin nulos) detienen el notebook; las de aviso solo se muestran.
**Por qué:** un `id` repetido o un flag nulo contaminan todo lo posterior; un dato fuera de rango suele ser un error local.
**Resultado esperado:** 0 en las dos listas (igual que el silver de pandas).

In [23]:
def n(cond):
    return df.filter(cond).count()

bloquean = {
    "id duplicados": df.count() - df.select("id").distinct().count(),
    "flags con nulos": n(F.col("price_missing").isNull() | F.col("price_outlier").isNull() | F.col("is_long_stay").isNull()),
}
avisan = {
    "latitud fuera de 42-44": n(F.col("latitude").isNull() | ~F.col("latitude").between(42, 44)),
    "longitud fuera de -4 a -1": n(F.col("longitude").isNull() | ~F.col("longitude").between(-4, -1)),
    "accommodates <= 0": n(F.col("accommodates") <= 0),
    "availability_30 > availability_365": n(F.col("availability_30") > F.col("availability_365")),
    "precio <= 0": n(F.col("price_num") <= 0),
    "first_review > last_review": n(F.col("first_review") > F.col("last_review")),
    "reseñas 12m > totales": n(F.col("number_of_reviews_ltm") > F.col("number_of_reviews")),
}
print("BLOQUEAN:", bloquean)
print("AVISAN  :", avisan)
assert all(v == 0 for v in bloquean.values()), f"Validación bloqueante fallida: {bloquean}"

StatementMeta(, 6e5bb2ae-5355-48cc-8022-3c7c72aabf33, 25, Finished, Available, Finished, False)

BLOQUEAN: {'id duplicados': 0, 'flags con nulos': 0}
AVISAN  : {'latitud fuera de 42-44': 0, 'longitud fuera de -4 a -1': 0, 'accommodates <= 0': 0, 'availability_30 > availability_365': 0, 'precio <= 0': 0, 'first_review > last_review': 0, 'reseñas 12m > totales': 0}


## 4d Escribir la tabla Delta en `lh_silver`
**Qué:** añado `ciudad` y `snapshot_date` y escribo `Tables/listings` en Delta, particionada por ciudad y fecha.
**Por qué:** `replaceWhere` reemplaza solo la partición de este snapshot, así que repetir la escritura no duplica y no toca los otros snapshots (idempotente).
**Resultado esperado:** euskadi / 2026-06-30 / 6.248 filas, también tras ejecutarla dos veces.

In [24]:
salida = (df
    .withColumn("ciudad", F.lit(CIUDAD))
    .withColumn("snapshot_date", F.to_date(F.lit(SNAPSHOT))))

destino = "abfss://revenue-intelligence-airbnb@onelake.dfs.fabric.microsoft.com/lh_silver.Lakehouse/Tables/listings"

(salida.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"ciudad = '{CIUDAD}' AND snapshot_date = '{SNAPSHOT}'")
    .partitionBy("ciudad", "snapshot_date")
    .save(destino))

spark.read.format("delta").load(destino).groupBy("ciudad", "snapshot_date").count().show()

StatementMeta(, 6e5bb2ae-5355-48cc-8022-3c7c72aabf33, 26, Finished, Available, Finished, False)

+-------+-------------+-----+
| ciudad|snapshot_date|count|
+-------+-------------+-----+
|euskadi|   2026-06-30| 6248|
+-------+-------------+-----+



In [2]:
destino = "abfss://revenue-intelligence-airbnb@onelake.dfs.fabric.microsoft.com/lh_silver.Lakehouse/Tables/listings"

StatementMeta(, c419f606-38b0-47e4-821d-d041b2c574dc, 4, Finished, Available, Finished, False)

In [3]:
spark.read.format("delta").load(destino).groupBy("ciudad", "snapshot_date").count().orderBy("ciudad", "snapshot_date").show()

StatementMeta(, c419f606-38b0-47e4-821d-d041b2c574dc, 5, Finished, Available, Finished, False)

+---------+-------------+-----+
|   ciudad|snapshot_date|count|
+---------+-------------+-----+
|barcelona|   2025-08-10|19325|
|barcelona|   2025-09-14|19410|
|  euskadi|   2026-06-30| 6248|
+---------+-------------+-----+



In [2]:
destino = "abfss://revenue-intelligence-airbnb@onelake.dfs.fabric.microsoft.com/lh_silver.Lakehouse/Tables/listings"
spark.read.format("delta").load(destino).groupBy("ciudad", "snapshot_date").count().orderBy("ciudad", "snapshot_date").show(30)

StatementMeta(, ace16ee5-1058-4017-985a-bf354e4f5d6b, 4, Finished, Available, Finished, False)

+---------+-------------+-----+
|   ciudad|snapshot_date|count|
+---------+-------------+-----+
|barcelona|   2025-08-10|19325|
|barcelona|   2025-09-14|19410|
|barcelona|   2025-10-17|15075|
|barcelona|   2025-11-17|17934|
|barcelona|   2025-12-14|18177|
|barcelona|   2026-01-18|17099|
|barcelona|   2026-02-18|12786|
|barcelona|   2026-03-21|16107|
|barcelona|   2026-04-20| 7277|
|barcelona|   2026-05-24|14399|
|barcelona|   2026-06-24|15293|
|  euskadi|   2025-07-30| 6506|
|  euskadi|   2025-08-31| 6494|
|  euskadi|   2025-09-29| 6334|
|  euskadi|   2025-10-31| 6181|
|  euskadi|   2025-11-30| 6171|
|  euskadi|   2025-12-31| 5460|
|  euskadi|   2026-01-28| 5430|
|  euskadi|   2026-02-27| 5512|
|  euskadi|   2026-03-31| 5111|
|  euskadi|   2026-04-30| 5050|
|  euskadi|   2026-05-31| 5387|
|  euskadi|   2026-06-30| 6248|
+---------+-------------+-----+

